# SDXL DreamBooth LoRA Training (Kaggle P100 16GB)

Zero-cost training pipeline using Kaggle's free P100 GPU.
- **GPU:** P100 16GB
- **Time:** ~2-3 hours per training iteration
- **Output:** LoRA weights uploaded to HuggingFace Hub

## Instructions
1. Set Settings → Accelerator → GPU P100 16GB
2. Set Settings → Internet → On
3. Add your HuggingFace token as a Kaggle secret
4. Upload your dataset to HuggingFace as a private dataset repo
5. Run cells in order

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q diffusers transformers accelerate safetensors
!pip install -q huggingface_hub
!pip install -q insightface onnxruntime-gpu
!pip install -q prodigyopt lion-pytorch

# Clone Kohya_ss sd-scripts
!git clone https://github.com/kohya-ss/sd-scripts.git /kaggle/working/sd-scripts
%cd /kaggle/working/sd-scripts
!pip install -r requirements.txt

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Cell 2: Configuration
import os

# HuggingFace token — set as Kaggle secret or paste here
HF_TOKEN = os.environ.get('HF_TOKEN', 'hf_your_token_here')
HF_USERNAME = 'your_username'
LORA_REPO_ID = f'{HF_USERNAME}/my-sdxl-lora'
DATASET_REPO_ID = f'{HF_USERNAME}/my-dataset'

# Training config
CONFIG = {
    'model': 'stabilityai/stable-diffusion-xl-base-1.0',
    'resolution': '1024,576',  # 16:9 for video input
    'trigger_word': 'ohwx person',
    'rank': 64,
    'alpha': 32,
    'lr': '1e-4',
    'steps': 1500,
    'batch_size': 1,
    'grad_accum': 4,
    'seed': 42,
}

print(f'LoRA repo: {LORA_REPO_ID}')
print(f'Dataset repo: {DATASET_REPO_ID}')
print(f'Training: {CONFIG["steps"]} steps, rank {CONFIG["rank"]}, lr {CONFIG["lr"]}')

In [ ]:
# Cell 3: Download dataset from HuggingFace
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id=DATASET_REPO_ID,
    repo_type='dataset',
    token=HF_TOKEN,
    local_dir='/kaggle/working/dataset'
)

import pathlib
images = list(pathlib.Path(dataset_path).glob('*.png')) + list(pathlib.Path(dataset_path).glob('*.jpg'))
captions = list(pathlib.Path(dataset_path).glob('*.txt'))
print(f'Dataset: {len(images)} images, {len(captions)} captions')

assert len(images) >= 10, f'Need at least 10 images, found {len(images)}'
assert len(captions) >= len(images), f'{len(images) - len(captions)} images missing captions'

In [ ]:
# Cell 4: Generate dataset_config.toml
toml_content = f'''[general]
resolution = [{CONFIG["resolution"].split(",")[0]}, {CONFIG["resolution"].split(",")[1]}]
shuffle_caption = true
keep_tokens = 1

[[datasets]]
batch_size = {CONFIG["batch_size"]}
enable_bucket = false

  [[datasets.subsets]]
  image_dir = "/kaggle/working/dataset"
  caption_extension = ".txt"
  num_repeats = 10
'''

with open('/kaggle/working/dataset_config.toml', 'w') as f:
    f.write(toml_content)
print('Generated dataset_config.toml')
print(toml_content)

In [ ]:
# Cell 5: Download SDXL base model
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id=CONFIG['model'],
    token=HF_TOKEN,
    local_dir='/kaggle/working/sdxl_base',
    allow_patterns=['*.json', '*.txt', '*.safetensors', '*.bin'],
)
print(f'SDXL model downloaded to: {model_path}')

In [ ]:
# Cell 6: Run Kohya_ss SDXL LoRA training
%cd /kaggle/working/sd-scripts

cmd = f'''accelerate launch --mixed_precision=fp16 sdxl_train_network.py \
    --pretrained_model_name_or_path={model_path} \
    --dataset_config=/kaggle/working/dataset_config.toml \
    --output_dir=/kaggle/working/output \
    --output_name=sdxl_lora \
    --train_batch_size={CONFIG["batch_size"]} \
    --gradient_accumulation_steps={CONFIG["grad_accum"]} \
    --learning_rate={CONFIG["lr"]} \
    --lr_scheduler=constant \
    --max_train_steps={CONFIG["steps"]} \
    --save_every_n_epochs=1 \
    --mixed_precision=fp16 \
    --save_model_as=safetensors \
    --network_module=networks.lora \
    --network_dim={CONFIG["rank"]} \
    --network_alpha={CONFIG["alpha"]} \
    --network_train_unet_only \
    --cache_text_encoder_outputs \
    --cache_latents \
    --gradient_checkpointing \
    --seed={CONFIG["seed"]}
'''

print('Starting training...')
print(f'Command: {cmd[:200]}...')
!{cmd}

In [ ]:
# Cell 7: Validate identity with ArcFace
import os
import numpy as np
from PIL import Image
from insightface.app import FaceAnalysis

app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

def get_embedding(img_path):
    img = np.array(Image.open(img_path).convert('RGB'))
    faces = app.get(img)
    if len(faces) == 0:
        return None
    return max(faces, key=lambda f: f.det_score).embedding

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Get reference embeddings
import pathlib
ref_dir = pathlib.Path('/kaggle/working/dataset')
ref_embs = []
for img_path in ref_dir.glob('*.png'):
    emb = get_embedding(str(img_path))
    if emb is not None:
        ref_embs.append(emb)

print(f'Reference embeddings: {len(ref_embs)}')

# Generate test images with LoRA
import torch
from diffusers import StableDiffusionXLPipeline

pipe = StableDiffusionXLPipeline.from_pretrained(
    model_path, torch_dtype=torch.float16, variant='fp16'
).to('cuda')
pipe.load_lora_weights('/kaggle/working/output/sdxl_lora.safetensors')

test_prompts = [
    f'{CONFIG["trigger_word"]}, front-facing portrait, studio lighting, photorealistic',
    f'{CONFIG["trigger_word"]}, side profile, outdoor, natural lighting, photorealistic',
    f'{CONFIG["trigger_word"]}, smiling, casual clothing, indoor, photorealistic',
]

sims = []
for i, prompt in enumerate(test_prompts):
    img = pipe(prompt, height=576, width=1024, num_inference_steps=30,
               cross_attention_kwargs={'scale': 0.9}).images[0]
    img.save(f'/kaggle/working/test_{i}.png')
    emb = get_embedding(f'/kaggle/working/test_{i}.png')
    if emb is not None and len(ref_embs) > 0:
        max_sim = max(cosine_sim(emb, ref) for ref in ref_embs)
        sims.append(max_sim)
        print(f'  Test {i}: similarity = {max_sim:.4f} {"✓" if max_sim >= 0.7 else "✗"}')

avg_sim = np.mean(sims) if sims else 0
print(f'\nAverage similarity: {avg_sim:.4f}')
if avg_sim >= 0.7:
    print('✓ PASS: Identity preservation is good')
else:
    print('✗ FAIL: Consider more steps, higher rank, or better dataset')

In [ ]:
# Cell 8: Upload LoRA to HuggingFace Hub
from huggingface_hub import HfApi, upload_file

api = HfApi(token=HF_TOKEN)

# Create repo if it doesn't exist
try:
    api.create_repo(repo_id=LORA_REPO_ID, repo_type='model', private=True)
    print(f'Created repo: {LORA_REPO_ID}')
except Exception:
    print(f'Repo already exists: {LORA_REPO_ID}')

# Upload LoRA weights
upload_file(
    path_or_fileobj='/kaggle/working/output/sdxl_lora.safetensors',
    path_in_repo='sdxl_lora.safetensors',
    repo_id=LORA_REPO_ID,
    token=HF_TOKEN,
)
print(f'✓ Uploaded LoRA weights to {LORA_REPO_ID}')
print(f'  Download in Colab: hf_hub_download(repo_id="{LORA_REPO_ID}", filename="sdxl_lora.safetensors")')